This notebook computes the geodesic distance between pairs of grocery stores.

In [2]:
## Modules

import geopandas as gpd
import pandas as pd
import numpy as np

import json
import urllib.request
import geopandas as gpd
import pandas as pd
import numpy as np
import geopy.distance

In [3]:
#geodesic distance definitions 
#
# NB: these functions say "walk" time for some reason

def get_dist(a, b):
    origin = str(a.y)+','+str(a.x)
    destination = str(b.y)+','+str(b.x)
    dist = geopy.distance.geodesic(origin, destination)
    return dist.meters
    
def get_walk_time(a,b):
    dist = get_dist(a,b)
    return dist

def calc_twalk_matrix(polls, neighbours):
    """
    Input:
    polls is a geodataframe storing locations as Points
    neighbours is a 0/1 array representing which locations we would like to compute distances between
    
    Returns: A matrix of walking times between locations
    """
    N = polls.shape[0]
    t_walk = np.zeros((N,N))
    for i, a in polls.iterrows():
        for j, b in polls.iterrows():
            if i != j:
                t_walk[i,j] = get_walk_time(a.geometry,b.geometry)
    return t_walk

In [4]:
groc_sites = gpd.read_file('Dallas/Food access_updated/geo_export_872fcb6c-fbde-4264-ae77-8858a604ed0e.shp' ) #import grocery store locations

In [5]:
nSites = groc_sites.shape[0] 
groc_neighbours = np.ones((nSites,nSites))

In [6]:
# Geodesic distance
dal_geodesic_distance_meters = calc_twalk_matrix(groc_sites, groc_neighbours)

In [7]:
walk_speed = 1.42 # meters/sec

In [8]:
dal_geodesic_distance_seconds = dal_geodesic_distance_meters/1.42 #compute "walk time" (in seconds) for geodesic distance

In [10]:
np.savez('dal_geodesic_distance_data', dal_geodesic_distance_meters = dal_geodesic_distance_meters,
         walk_speed = walk_speed, dal_geodesic_distance_seconds = dal_geodesic_distance_seconds)